From the GNN Node Level experiemnts. I know that the features I want to be using are:
- ["adv", "hac", "hbam", "c", "asb"] from MQNs
- One hot Encoded adducts fromm: "[M+H]+", "[M+Na]+", "[M-H]-", "[M+NH4]+", "[M+K]+", "[M+H-H2O]+", "[M+HCOO]-", "[M+CH3COO]-", "[M+Na-2H]-"
- mz

Now we are going to try rebuild a GNN from the graph level with 3d molecular descriptors

In [ ]:
import sqlite3
import pandas as pd 

In [2]:
conn = sqlite3.connect("/Users/reubensantoso/Xu_Lab_Files/c3s_neural_network/C3S.db")
master_df = pd.read_sql_query("SELECT * FROM master", conn)
mqn_df = pd.read_sql_query("SELECT * FROM mqns", conn)
fngr_df = pd.read_sql_query("SELECT * FROM fingerprints", conn)
conn.close()

master_df = master_df.dropna()
print(master_df.shape)
print(mqn_df.shape)
print(fngr_df.shape)

(15187, 12)
(15187, 43)
(15187, 1025)


In [3]:
master_df.head(2)

,g_id,name,adduct,mass,z,mz,ccs,smi,chem_class_label,src_tag,ccs_type,ccs_method
0,CCSBASE_C4B6CF0FE6,1-Methylnicotinamide,[M]+,137.0715,1,137.0715,126.4,C[N+]1=CC=CC(=C1)C(=O)N,small molecule,zhou1016,DT,"single field, calibrated with Agilent tune mix..."
1,CCSBASE_D0DE2590C2,7-Methylguanosine,[M]+,298.1151,1,298.1151,166.5,CN1C=[N+](C2=C1C(=O)N=C(N2)N)[C@H]3[C@@H]([C@@...,small molecule,zhou1016,DT,"single field, calibrated with Agilent tune mix..."


# Building Features

In [4]:
from sklearn.preprocessing import OneHotEncoder

/opt/miniconda3/envs/database_clean/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [5]:
c3s_df = pd.DataFrame()

# adding mz into c3s
c3s_df = master_df[["g_id", "mz", "smi"]].copy()

# adding one hot encoded adducts into c3s
adduct_groups = [
    "[M+H]+", "[M+Na]+", "[M-H]-", "[M+NH4]+", "[M+K]+",
    "[M+H-H2O]+", "[M+HCOO]-", "[M+CH3COO]-", "[M+Na-2H]-"
]

adduct_encoding = OneHotEncoder(
    sparse_output=False,
    categories='auto', 
    handle_unknown="infrequent_if_exist",
    min_frequency=30 #needs to be above 5 data samples to be encoded. if below gets grouped to others
)

adduct_encoded = adduct_encoding.fit_transform(master_df[['adduct']])
adduct_encoded_df = pd.DataFrame(
    adduct_encoded, 
    columns=adduct_encoding.get_feature_names_out(['adduct'])
)
adduct_encoded_df["g_id"] = master_df["g_id"].values

# Merge encoded adducts by g_id
c3s_df = c3s_df.merge(adduct_encoded_df, on="g_id", how="left")

# Add MQN features by merging on g_id
c3s_df = c3s_df.merge(mqn_df[["g_id", "adv", "hac", "hbam", "c", "asb"]], on="g_id", how="left")

# Final result
print(c3s_df.shape)
c3s_df.head(2)


(15187, 22)


,g_id,mz,smi,adduct_[2M-H]-,adduct_[M+2Na-H]+,adduct_[M+CH3COO]-,adduct_[M+Cl]-,adduct_[M+H-H2O]+,adduct_[M+HCOO]-,adduct_[M+H]+,...,adduct_[M+Na-2H]-,adduct_[M+Na]+,adduct_[M-H]-,adduct_[M]+,adduct_infrequent_sklearn,adv,hac,hbam,c,asb
0,CCSBASE_C4B6CF0FE6,137.0715,C[N+]1=CC=CC(=C1)C(=O)N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0,10,3,7,1
1,CCSBASE_D0DE2590C2,298.1151,CN1C=[N+](C2=C1C(=O)N=C(N2)N)[C@H]3[C@@H]([C@@...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1,21,7,11,2


In [6]:
target_df = master_df[["g_id","ccs"]]
target_df.shape

(15187, 2)

# Pytorch Setup

In [7]:
from sklearn.model_selection import train_test_split
import torch
from torch_geometric.data import Data
from rdkit import Chem

In [8]:
use_subset = False

if use_subset:
    c3s_df = c3s_df.sample(n=800, random_state=42).reset_index(drop=True)
    fngr_df = fngr_df[fngr_df["g_id"].isin(c3s_df["g_id"])]


In [9]:
print(c3s_df.shape)
print(fngr_df.shape)

(15187, 22)
(15187, 1025)


In [10]:
from torch_geometric.data import Data
from rdkit import Chem
import torch

def mol_to_graph(smiles, features, fingerprint, target):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  # skip invalid molecules

    # Create dummy per-atom features if needed
    num_atoms = mol.GetNumAtoms()
    atom_features = torch.tensor([[f] * num_atoms for f in features], dtype=torch.float).T

    # Edge index from bond connectivity
    edges = []
    for bond in mol.GetBonds():
        a, b = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edges.append([a, b])
        edges.append([b, a])
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    fingerprint = torch.tensor(fingerprint, dtype=torch.float)
    y = torch.tensor([target], dtype=torch.float)

    return Data(x=atom_features, edge_index=edge_index, y=y, fingerprints=fingerprint)

# --- Build dataset ---
dataset = []
for i, row in c3s_df.iterrows():
    try:
        smiles = row['smi']  
        features = row.drop(['g_id', 'smi']).astype(float).values
        fingerprint = fngr_df[fngr_df['g_id'] == row['g_id']].drop(columns=["g_id"]).astype(float).values.flatten()
        target = target_df[target_df['g_id'] == row['g_id']]['ccs'].values[0]

        data = mol_to_graph(smiles, features, fingerprint, target)
        if data is not None:
            dataset.append(data)

    except Exception as e:
        print(f"Skipping index {i} due to error: {e}")


In [11]:
print(dataset[:3])

[Data(x=[10, 20], edge_index=[2, 20], y=[1], fingerprints=[1024]), Data(x=[21, 20], edge_index=[2, 46], y=[1], fingerprints=[1024]), Data(x=[27, 20], edge_index=[2, 54], y=[1], fingerprints=[1024])]


In [ ]:
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42)
train_data, val_data = train_test_split(train_data, test_size=0.1, random_state=42)

batch_size = 64
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size)
test_loader = DataLoader(test_data, batch_size=batch_size)


# Making GNN

In [13]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


Your molecule enters as a graph.
GIN learns its chemical structure, 3D descriptors add geometry info,
then an MLP maps that combined info to predict your cross-collisional value directly.

This architecture neatly integrates graph-learned chemistry and expert-crafted shape insights, providing a balanced yet powerful modeling approach for your task.

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINConv, global_mean_pool, BatchNorm
from torch_geometric.data import Data

class GIN_Fingerprint_Net(nn.Module):
    """
    GIN graph encoder + concatenated graph-level fingerprint → MLP → scalar output.
    Accepts either:
        • a PyG Data/Batch object   → model(data)
        • four separate tensors     → model(x, edge_index, batch, fingerprints)
    """
    def __init__(self, node_feature_dim: int, fingerprint_dim: int, dropout_p: float = 0.1):
        super().__init__()
        self.fp_dim = fingerprint_dim
        self.dropout  = dropout_p

        self.conv = GINConv(nn.Sequential(
            #Linear Layer one
            nn.Linear(node_feature_dim, 256),
            nn.ReLU(),
            
            # dont dropout during massage pooling layers    
                
            #Linear Layer two
            nn.Linear(256, 128)
        ))
        
        self.bn   = BatchNorm(128)

        self.fc1 = nn.Linear(128 + fingerprint_dim, 64)
        self.dropout = nn.Dropout(0.1)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, *inputs):
        if len(inputs) == 1:
            data = inputs[0]
            x, edge_index, batch, fp = data.x, data.edge_index, data.batch, data.fingerprints
        else:
            x, edge_index, batch, fp = inputs

        x = F.relu(self.bn(self.conv(x,edge_index)))
        x = global_mean_pool(x, batch)
        
        if fp.dim() == 1:                       # e.g. [N*fp_dim]
            fp = fp.view(-1, self.fp_dim)       # -> [N, fp_dim]

        x = torch.cat([x, fp], dim=1)
        
        x = F.relu(self.fc1(x))
        
        x = self.dropout(x)
        
        return self.fc2(x).squeeze(1)       
    

# Training

In [15]:
import numpy as np
import copy

In [16]:
sample          = dataset[0]
node_feat_dim   = sample.x.size(-1)
fingerprint_dim = sample.fingerprints.size(-1)

model = GIN_Fingerprint_Net(
    node_feature_dim=node_feat_dim,
    fingerprint_dim=fingerprint_dim,
    dropout_p = 0.1
).to(device)
          
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.05,  
    weight_decay= 1e-4 
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.5, 
    patience=10,
    min_lr=1e-6
)

criterion  = torch.nn.MSELoss()      
eps        = 1e-8                    # for safe division in MRE


In [17]:
num_epochs = 400
early_patience = 35
patience = 0

best_val_mre   = float('inf')
best_train_mre   = float('inf')
best_model_wts = None

for epoch in range(1, num_epochs + 1):

    # ── train ─────────────────────────────────────────────────────────
    model.train()
    train_losses, train_mres = [], []

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        preds = model(batch)
        loss  = criterion(preds, batch.y)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_mres.append(
            torch.mean(100. * torch.abs(preds - batch.y) / (batch.y + eps)).item()
        )

    # ── validate ─────────────────────────────────────────────────────
    model.eval()
    val_losses, val_mres = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            preds = model(batch)
            loss  = criterion(preds, batch.y)

            val_losses.append(loss.item())
            val_mres.append(
                torch.mean(100. * torch.abs(preds - batch.y) / (batch.y + eps)).item()
            )

    # step LR scheduler with mean validation loss
    scheduler.step(np.mean(val_losses))

    # ── early-stopping logic ──────────────────────────────────────────
    curr_train_mre = np.mean(train_mres)
    if curr_train_mre < best_train_mre:
        best_train_mre = curr_train_mre
    
    curr_val_mre = np.mean(val_mres)
    if curr_val_mre < best_val_mre:
        best_val_mre = curr_val_mre
        best_model_wts = model.state_dict()
        patience = 0                     # reset counter
    else:
        patience += 1
        if patience >= early_patience:
            print(f"\nEarly stopping at epoch {epoch}")
            break

    # ── epoch report ─────────────────────────────────────────────────
    print(
        f"Epoch {epoch:03d} │ "
        f"Train-Loss {np.mean(train_losses):.4f} │ Train-MRE {np.mean(train_mres):.2f}% │ "
        f"Val-Loss {np.mean(val_losses):.4f} │ Val-MRE {curr_val_mre:.2f}%"
    )

# restore best model
if best_model_wts is not None:
    model.load_state_dict(best_model_wts)


print(f"Best Train MRE: {best_train_mre}")
print(f"Best Val MRE: {best_val_mre}")

Epoch 001 │ Train-Loss 2126.5706 │ Train-MRE 13.58% │ Val-Loss 330.6955 │ Val-MRE 6.34%
Epoch 002 │ Train-Loss 488.5861 │ Train-MRE 8.06% │ Val-Loss 409.8701 │ Val-MRE 8.83%
Epoch 003 │ Train-Loss 503.6413 │ Train-MRE 8.20% │ Val-Loss 693.5995 │ Val-MRE 10.20%
Epoch 004 │ Train-Loss 492.8881 │ Train-MRE 8.09% │ Val-Loss 248.6247 │ Val-MRE 6.01%
Epoch 005 │ Train-Loss 476.5930 │ Train-MRE 7.90% │ Val-Loss 180.6263 │ Val-MRE 4.77%
Epoch 006 │ Train-Loss 531.7237 │ Train-MRE 8.46% │ Val-Loss 268.5385 │ Val-MRE 6.56%
Epoch 007 │ Train-Loss 494.6108 │ Train-MRE 8.22% │ Val-Loss 163.7876 │ Val-MRE 4.66%
Epoch 008 │ Train-Loss 477.6837 │ Train-MRE 8.01% │ Val-Loss 195.0573 │ Val-MRE 4.78%
Epoch 009 │ Train-Loss 457.6970 │ Train-MRE 7.84% │ Val-Loss 778.9680 │ Val-MRE 11.36%
Epoch 010 │ Train-Loss 565.4812 │ Train-MRE 8.66% │ Val-Loss 233.0043 │ Val-MRE 5.74%
Epoch 011 │ Train-Loss 408.5592 │ Train-MRE 7.39% │ Val-Loss 174.7709 │ Val-MRE 4.46%
Epoch 012 │ Train-Loss 470.6174 │ Train-MRE 7.92% 

# See Test Performance

In [18]:
print(f"Best Model Weights: {best_model_wts}")

Best Model Weights: OrderedDict([('conv.eps', tensor([0.])), ('conv.nn.0.weight', tensor([[-1.9753e-39, -1.1575e-42, -8.8349e-41,  ..., -6.1221e-40,
          1.4093e-41, -2.9846e-40],
        [-3.2369e-27,  3.1027e-38, -4.4624e-39,  ..., -1.8436e-38,
         -1.9771e-37, -4.3662e-38],
        [-4.3142e-08,  6.4642e-40,  3.1992e-40,  ..., -2.2317e-37,
          2.2966e-37, -2.2964e-37],
        ...,
        [-4.7088e-39, -4.7050e-38, -1.7973e-41,  ..., -5.1687e-40,
         -1.7703e-40, -4.9731e-40],
        [-9.1606e-37, -4.1583e-40, -1.5055e-39,  ..., -9.3714e-39,
          2.0111e-39, -1.0906e-38],
        [-9.4803e-39,  1.4304e-38,  1.0001e-40,  ...,  5.7315e-39,
         -2.7562e-39,  4.8060e-39]])), ('conv.nn.0.bias', tensor([-1.1536e-38,  6.8009e-38, -5.8594e-41,  7.7825e-40, -2.8821e-40,
        -5.1442e-38,  6.1764e-40, -3.2162e-38, -6.6388e-40, -4.8847e-40,
        -5.0135e-40, -8.3946e-41, -1.2030e-40,  8.3267e-41,  2.1513e-40,
         3.9167e-38, -9.0443e-41,  9.6622e-41,

In [ ]:
model.eval()
test_losses, test_mres = [], []
with torch.no_grad():
    
    for batch in test_loader:
        batch = batch.to(device)
        preds = model(batch)                    
        test_losses.append(criterion(preds, batch.y).item())
        abs_err = torch.abs(preds - batch.y)
        test_mres.append(torch.mean(100. * abs_err / (batch.y + eps)).item())

print(f"\nTest │ Loss {np.mean(test_losses):.4f} │ MRE {np.mean(test_mres):.2f}%")


Test │ Loss 278.6586 │ MRE 2.70%


In [24]:
import pandas as pd

# Extract the necessary data
results = []
for i, batch in enumerate(test_loader):
    batch = batch.to(device)
    with torch.no_grad():
        preds = model(batch)
    for j in range(batch.num_graphs):
        g_id = test_data[i * batch_size + j].y.item()  # Original CCS
        predicted_ccs = preds[j].item()  # Predicted CCS
        entry_name = test_data[i * batch_size + j].g_id  # Entry name
        results.append({"Entry Name": entry_name, "Original CCS": g_id, "Predicted CCS": predicted_ccs})

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Save to CSV
results_df.to_csv("predicted_ccs_results.csv", index=False)
print("Results saved to predicted_ccs_results.csv")

AttributeError: 'GlobalStorage' object has no attribute 'g_id'

# Visuals

In [20]:
# with torch.no_grad():
#     y_pred_train = best_gnn_model(data_train_val) #predicted values from train
#     y_pred_test = best_gnn_model(data_test) #predicted values from test

# #convert to numpy
# y_train_np = data_train_val.y.detach().cpu().numpy()
# y_test_np = data_test.y.detach().cpu().numpy()
# y_pred_train_np = y_pred_train.detach().cpu().numpy()
# y_pred_test_np = y_pred_test.detach().cpu().numpy()

In [21]:
# from typing import Dict, List
# from matplotlib.gridspec import GridSpec
# from numpy import typing as npt
# from sklearn.metrics import r2_score, mean_squared_error

# def compute_metrics(y: npt.NDArray[np.float64],
#                     y_pred: npt.NDArray[np.float64]
#                     ) -> Dict[str, float | List[float]] :
#     """
#     compute a standard array of metrics between a set of predicted and reference values

#     - R-squared (R2)
#     - root mean squared error (RMSE)
#     - mean absolute error (MAE)
#     - median absolute erre (MDAE)
#     - mean relative error % (MRE)
#     - median relative error % (MDRE)
#     - cumulative error distribution at the <1, <3, <5, and <10% levels (CE135A)

#     Parameters
#     ----------
#     y : ``numpy.ndarray(float)``
#     y_pred : ``numpy.ndarray(float)``
#         arrays of reference and predicted values

#     Returns
#     -------
#     summary : ``dict(...)``
#         dict with set of metrics
#     """
#     abs_y_err = np.abs(y_pred - y)
#     r2 = r2_score(y, y_pred)
#     mae = np.mean(abs_y_err)
#     mdae = np.median(abs_y_err)
#     mre = np.mean(100. * abs_y_err / y)
#     mdre = np.median(100. * abs_y_err / y)
#     rmse = np.sqrt(mean_squared_error(y, y_pred))
#     y_err_percent = 100. * abs_y_err / y
#     cum_err = np.cumsum(np.histogram(y_err_percent, [_ for _ in range(101)])[0])
#     cum_err = 100. * cum_err / np.sum(cum_err)
#     ce1, ce3, ce5, ceA = cum_err[0], cum_err[2], cum_err[4], cum_err[9]
#     return {
#         'R2': r2, 'MAE': mae, 'MDAE': mdae, 'MRE': mre, 'MDRE': mdre,
#         'RMSE': rmse, 'CE135A': [ce1, ce3, ce5, ceA]
#     }


# def compute_metrics_train_test(y_train: npt.NDArray[np.float64],
#                                y_test: npt.NDArray[np.float64],
#                                y_pred_train: npt.NDArray[np.float64],
#                                y_pred_test: npt.NDArray[np.float64]
#                                ) -> Dict[str, Dict[str, float | List[float]]] :
#     """
#     computes a standard set of performance metrics from separate predictions
#     for a training and test dataset
#     - training and test set R-squared (R2)
#     - training and test set root mean squared error (RMSE)
#     - training and test set mean absolute error (MAE)
#     - training and test set median absolute error (MDAE)
#     - training and test set mean relative error (MRE)
#     - training and test set median relative error (MDRE)
#     - cumulative error distribution at the <1, <3, <5, and <10% levels
#         for training and test set (CE135A)

#     Parameters
#     ----------
#     y_train : ``numpy.ndarray(float)``
#     y_test : ``numpy.ndarray(float)``
#     y_pred_train : ``numpy.ndarray(float)``
#     y_pred_test : ``numpy.ndarray(float)``
#         arrays of reference and predicted values for training and test datasets

#     Returns
#     -------
#     summary : ``dict(...)``
#         dict with set of metrics for training and test datasets
#     """
#     summary = {}
#     for y, y_pred, lbl in [(y_train, y_pred_train, "train"), (y_test, y_pred_test, "test")]:
#         # compute metrics
#         summary[lbl] = compute_metrics(y, y_pred)
#     return summary

# def train_test_summary_figure(summary: Dict[str, Dict[str, float | List[float]]],
#                               fig_name: str,
#                               r2_range=[0.95, 1.]):
#     """
#     produces a summary figure displaying the results from `compute_metrics_train_test`

#     Parameters
#     ----------
#     summary : ``dict(...)``
#         summary dict returned from `compute_metrics_train_test`
#     fig_name : ``str``
#         file name to save the generated plot under
#     r2_range : ``list(float))``, default=[0.95, 1.]]
#         lower and upper bounds of R-squared y axis
#     """
#     fig = plt.figure(figsize=(5, 3))
#     gs = GridSpec(1, 4, figure=fig, width_ratios=[1.2, 3, 2, 5])

#     # R-squared
#     ax1 = fig.add_subplot(gs[0])
#     rsq_trn = summary['train']['R2']
#     rsq_tst = summary['test']['R2']
#     w1 = 0.15
#     ax1.bar([1 - w1 / 2, 1 + w1 / 2], [rsq_trn, rsq_tst], color=['b', 'r'], width=w1)
#     for d in ['top', 'right']:
#         ax1.spines[d].set_visible(False)
#     ax1.set_xticks([])
#     ax1.set_ylabel(r'R$^2$')
#     ax1.set_ylim(r2_range)
#     ax1.set_xlim([0.75, 1.25])

#     # MAE, MDAE and RMSE
#     ax2 = fig.add_subplot(gs[1])
#     mae_trn = summary['train']['MAE']
#     mae_tst = summary['test']['MAE']
#     mdae_trn = summary['train']['MDAE']
#     mdae_tst = summary['test']['MDAE']
#     mse_trn = summary['train']['RMSE']
#     mse_tst = summary['test']['RMSE']
#     ax2.bar([0.875, 1.125], [mae_trn, mae_tst], color=['b', 'r'], width=0.25)
#     ax2.bar([1.875, 2.125], [mdae_trn, mdae_tst], color=['b', 'r'], width=0.25)
#     ax2.bar([2.875, 3.125], [mse_trn, mse_tst], color=['b', 'r'], width=0.25)
#     for d in ['top', 'right']:
#         ax2.spines[d].set_visible(False)
#     ax2.set_xticks([1, 2, 3])
#     ax2.set_xticklabels(['MAE', 'MDAE', 'RMSE'], rotation='vertical')
#     ax2.set_ylabel(r'CCS (Å$^2$)')

#     # CE135A
#     ax3 = fig.add_subplot(gs[3])
#     x1 = [_ - 0.125 for _ in range(1, 5)]
#     y1 = [100. * summary['train']['CE135A'][i] for i in range(4)]
#     x2 = [_ + 0.125 for _ in range(1, 5)]
#     y2 = [100. * summary['test']['CE135A'][i] for i in range(4)]
#     ax3.bar(x1, y1, color='b', width=0.25)
#     ax3.bar(x2, y2, color='r', width=0.25)
#     for d in ['top', 'right']:
#         ax3.spines[d].set_visible(False)
#     ax3.set_xlabel('pred. error (%)')
#     ax3.set_xticks([1, 2, 3, 4])
#     ax3.set_xticklabels(['<1', '<3', '<5', '<10'])
#     ax3.set_ylabel('proportion (%)')
#     # between axes 2 and 3

#     # MRE, MDRE
#     ax23 = fig.add_subplot(gs[2])
#     mre_trn = summary['train']['MRE']
#     mre_tst = summary['test']['MRE']
#     mdre_trn = summary['train']['MDRE']
#     mdre_tst = summary['test']['MDRE']
#     w23 = 0.25
#     ax23.bar([1 - w23 / 2, 1 + w23 / 2], [mre_trn, mre_tst], color=['b', 'r'], width=w23)
#     ax23.bar([2 - w23 / 2, 2 + w23 / 2], [mdre_trn, mdre_tst], color=['b', 'r'], width=w23)
#     for d in ['top', 'right']:
#         ax23.spines[d].set_visible(False)
#     ax23.set_xticks([1, 2])
#     ax23.set_xticklabels(['MRE', 'MDRE'], rotation='vertical')
#     ax23.set_ylabel('%')
#     plt.tight_layout()
#     plt.savefig(fig_name, dpi=400, bbox_inches='tight')

In [22]:
# summary_new = compute_metrics_train_test(y_train_np, y_test_np, y_pred_train_np, y_pred_test_np)
# train_test_summary_figure(summary_new, "metrics_gnn.png")

In [23]:
# for key, metrics_dict in summary_new.items():
#     print(key)  # This prints "train" or "test"
#     for metric_name, value in metrics_dict.items():
#         print(f"{metric_name}: {value}\n")